<a href="https://colab.research.google.com/github/risinukaviranga/ci-cd-build-failure-prediction/blob/main/Data_processing_and_data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Setting up environment

In [22]:
!pip install -q duckdb pandas numpy pyarrow scikit-learn

In [23]:
import duckdb
import pandas as pd
import numpy as np
import os

from google.colab import drive
drive.mount('/content/drive')

DATASET_PATH = "/content/drive/MyDrive/PipelineX/travistorrent.csv"

REPORT_PATH = "/content/drive/MyDrive/PipelineX/reports/preprocessing"

OUTPUT_PATH = "/content/drive/MyDrive/PipelineX/data/processed"

os.makedirs(REPORT_PATH, exist_ok=True)
os.makedirs(OUTPUT_PATH, exist_ok=True)

TEMP_PATH = "/content/duckdb_temp"
os.makedirs(TEMP_PATH, exist_ok=True)

con = duckdb.connect("/content/pipelinex.duckdb")

con.execute("SET memory_limit='4GB'")
con.execute("SET threads=2")
con.execute(f"SET temp_directory='{TEMP_PATH}'")

csv_path = DATASET_PATH.replace("'", "''")

con.execute(f"""
    CREATE OR REPLACE VIEW travis AS
    SELECT *
    FROM read_csv_auto('{csv_path}', sample_size=100000)
""")

print("Dataset loaded successfully!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset loaded successfully!


Inspect target variable/values

In [24]:
status_report = con.execute("""
    SELECT
        tr_status,
        COUNT(*) AS total_records
    FROM travis
    GROUP BY tr_status
    ORDER BY total_records DESC
""").df()

display(status_report)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,tr_status,total_records
0,passed,2810783
1,failed,751531
2,errored,279483
3,canceled,40192
4,started,3


In [25]:
status_report.to_csv(
    f"{REPORT_PATH}/original_status_values.csv",
    index=False
)

sucessfill build=0
failed build=1

Handle invalid records

In [26]:
invalid_report = con.execute("""
    SELECT
        COUNT(*) AS total_records,

        SUM(
            CASE WHEN tr_build_id IS NULL
            THEN 1 ELSE 0 END
        ) AS missing_build_ids,

        SUM(
            CASE WHEN tr_status IS NULL
            THEN 1 ELSE 0 END
        ) AS missing_status,

        SUM(
            CASE
                WHEN tr_status IS NOT NULL
                 AND TRIM(CAST(tr_status AS VARCHAR)) = ''
                THEN 1 ELSE 0
            END
        ) AS empty_status

    FROM travis
""").df()

display(invalid_report)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_records,missing_build_ids,missing_status,empty_status
0,3881992,0.0,0.0,0.0


In [27]:
invalid_status_records = con.execute("""
    SELECT *
    FROM travis
    WHERE tr_status IS NULL
       OR TRIM(CAST(tr_status AS VARCHAR)) = ''
""").df()

display(invalid_status_records.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,tr_build_id,gh_project_name,gh_is_pr,gh_pr_created_at,gh_pull_req_num,gh_lang,git_merged_with,git_branch,gh_num_commits_in_push,gh_commits_in_push,...,tr_log_num_test_suites_run,tr_log_num_test_suites_ok,tr_log_num_test_suites_failed,tr_log_tests_failed,tr_log_testduration,tr_log_buildduration,tr_original_commit,tr_duration,tr_status,tr_jobs


Investigate duplicate records

In [28]:
duplicate_jobs = con.execute("""
    SELECT
        tr_job_id,
        COUNT(*) AS occurrences
    FROM travis
    WHERE tr_job_id IS NOT NULL
    GROUP BY tr_job_id
    HAVING COUNT(*) > 1
    ORDER BY occurrences DESC
""").df()

display(duplicate_jobs.head(20))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,tr_job_id,occurrences


check duplicate rows

In [29]:
duplicate_summary = con.execute("""
    SELECT
        COUNT(*) AS total_records,
        COUNT(DISTINCT tr_job_id) AS distinct_job_ids,
        COUNT(DISTINCT tr_build_id) AS distinct_build_ids
    FROM travis
""").df()

display(duplicate_summary)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_records,distinct_job_ids,distinct_build_ids
0,3881992,3881992,925897


Handle missing values

In [30]:
schema = con.execute("DESCRIBE SELECT * FROM travis").df()

columns = schema["column_name"].tolist()

missing_queries = []

for column in columns:

    safe_column = '"' + column.replace('"', '""') + '"'

    missing_queries.append(f"""
        SELECT
            '{column}' AS column_name,

            SUM(
                CASE
                    WHEN {safe_column} IS NULL
                    THEN 1 ELSE 0
                END
            ) AS missing_count

        FROM travis
    """)

missing_sql = " UNION ALL ".join(missing_queries)

missing_report = con.execute(missing_sql).df()

total_records = con.execute(
    "SELECT COUNT(*) FROM travis"
).fetchone()[0]

missing_report["missing_percentage"] = (
    missing_report["missing_count"] / total_records * 100
)

missing_report = missing_report.sort_values(
    "missing_percentage",
    ascending=False
)

display(missing_report.head(30))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,column_name,missing_count,missing_percentage
59,tr_log_tests_failed,3724807.0,95.950919
6,git_merged_with,3061250.0,78.857710
18,tr_virtual_merged_into,3061250.0,78.857710
49,tr_log_frameworks,1542320.0,39.730118
51,tr_log_bool_tests_failed,1474485.0,37.982690
54,tr_log_num_tests_run,1474485.0,37.982690
53,tr_log_num_tests_failed,1474485.0,37.982690
52,tr_log_num_tests_ok,1474485.0,37.982690
55,tr_log_num_tests_skipped,1474485.0,37.982690
11,git_prev_built_commit,1253942.0,32.301509


In [31]:
missing_report.to_csv(
    f"{REPORT_PATH}/missing_value_report.csv",
    index=False
)

Check the data types

In [32]:
data_types = con.execute("""
    DESCRIBE SELECT * FROM travis
""").df()

display(data_types)

,column_name,column_type,null,key,default,extra
0,tr_build_id,BIGINT,YES,None,None,None
1,gh_project_name,VARCHAR,YES,None,None,None
2,gh_is_pr,BOOLEAN,YES,None,None,None
3,gh_pr_created_at,VARCHAR,YES,None,None,None
4,gh_pull_req_num,VARCHAR,YES,None,None,None
...,...,...,...,...,...,...
61,tr_log_buildduration,VARCHAR,YES,None,None,None
62,tr_original_commit,VARCHAR,YES,None,None,None
63,tr_duration,VARCHAR,YES,None,None,None
64,tr_status,VARCHAR,YES,None,None,None


In [33]:
data_types.to_csv(
    f"{REPORT_PATH}/preprocessing_schema.csv",
    index=False
)

In [34]:
important_columns = [
    "tr_build_id",
    "tr_job_id",
    "tr_status",
    "gh_is_pr",
    "gh_build_started_at",
    "gh_lang"
]

for column in important_columns:

    if column in columns:

        print(f"\nColumn: {column}")

        result = con.execute(f"""
            SELECT
                "{column}"
            FROM travis
            LIMIT 5
        """).df()

        display(result)


Column: tr_build_id


,tr_build_id
0,3154
1,3154
2,3154
3,3154
4,3154



Column: tr_job_id


,tr_job_id
0,3160
1,3161
2,3162
3,3163
4,3164



Column: tr_status


,tr_status
0,passed
1,passed
2,passed
3,passed
4,passed


... (2 outputs left out)


Column: gh_build_started_at


,gh_build_started_at
0,2011-04-16 11:24:39
1,2011-04-16 11:24:39
2,2011-04-16 11:24:39
3,2011-04-16 11:24:39
4,2011-04-16 11:24:39



Column: gh_lang


,gh_lang
0,ruby
1,ruby
2,ruby
3,ruby
4,ruby


In [35]:
timestamp_report = con.execute("""
    SELECT
        COUNT(*) AS total_records,

        COUNT(
            TRY_CAST(gh_build_started_at AS TIMESTAMP)
        ) AS valid_timestamps

    FROM travis
""").df()

display(timestamp_report)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_records,valid_timestamps
0,3881992,3881992


encode the target variable

In [36]:
STATUS_MAPPING = {
    "passed": 0,
    "failed": 1,
    "errored": 1
}

In [37]:
# Convert the confirmed mapping into SQL CASE conditions.

case_conditions = []

for status, label in STATUS_MAPPING.items():

    safe_status = status.replace("'", "''")

    case_conditions.append(
        f"WHEN LOWER(TRIM(CAST(tr_status AS VARCHAR))) "
        f"= '{safe_status}' THEN {label}"
    )

case_sql = "\n".join(case_conditions)

print(case_sql)

WHEN LOWER(TRIM(CAST(tr_status AS VARCHAR))) = 'passed' THEN 0
WHEN LOWER(TRIM(CAST(tr_status AS VARCHAR))) = 'failed' THEN 1
WHEN LOWER(TRIM(CAST(tr_status AS VARCHAR))) = 'errored' THEN 1


Remove data leakage

Potential leakage fields to investigate include:
- Build completion timestamps.
- Build duration.
- Test results generated after execution.
- Log-analysis results.
- Final build outcome fields.
- Post-build error or failure information.

Validate the cleaned dataset

In [38]:
con.execute("SHOW TABLES").df()


,name
0,travis


In [42]:
# 1. Create the cleaned_dataset table containing the mapped build_failure target and removing invalid status values like 'canceled' or 'started'
con.execute("""
    CREATE OR REPLACE TABLE cleaned_dataset AS
    SELECT
        *,
        CASE
            WHEN LOWER(TRIM(CAST(tr_status AS VARCHAR))) = 'passed' THEN 0
            WHEN LOWER(TRIM(CAST(tr_status AS VARCHAR))) = 'failed' THEN 1
            WHEN LOWER(TRIM(CAST(tr_status AS VARCHAR))) = 'errored' THEN 1
            ELSE NULL
        END AS build_failure
    FROM travis
    WHERE LOWER(TRIM(CAST(tr_status AS VARCHAR))) IN ('passed', 'failed', 'errored')
""")

# 2. Run the validation query on the newly created table
validation = con.execute("""
    SELECT
        COUNT(*) AS total_records,

        SUM(
            CASE WHEN build_failure IS NULL
            THEN 1 ELSE 0 END
        ) AS missing_targets,

        COUNT(DISTINCT build_failure)
            AS distinct_target_classes

    FROM cleaned_dataset
""").df()

display(validation)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_records,missing_targets,distinct_target_classes
0,3841797,0.0,2


Exporting my cleaned dataset

In [43]:
# Example export after the cleaned_dataset table is created.

OUTPUT_FILE = f"{OUTPUT_PATH}/cleaned_dataset.parquet"

con.execute(f"""
    COPY cleaned_dataset
    TO '{OUTPUT_FILE}'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

print("Cleaned dataset exported successfully!")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Cleaned dataset exported successfully!


# Milestone 03 – Data Preprocessing Findings

## 1. Original Dataset
- Original number of records:
- Original number of columns:

## 2. Invalid Records
- Records with missing target:
- Records excluded:

## 3. Duplicate Analysis
- Repeated job IDs:
- Duplicate handling decision:

## 4. Missing Values
- Features with highest missing percentages:
- Imputation decisions:

## 5. Data Types
- Columns requiring conversion:
- Timestamp validation:

## 6. Target Encoding
- Successful status values:
- Failed status values:
- Excluded status values:

## 7. Data Leakage
- Excluded identifiers:
- Excluded post-build features:
- Features retained for prediction:

## 8. Final Dataset
- Final record count:
- Final feature count:
- Target distribution:
- Export location:

## 9. Conclusion
- Main preprocessing decisions:
- Remaining data quality concerns: